In [1]:
#Install all required libraries
!pip install diffusers transformers accelerate gradio torch torchvision --quiet
!pip install xformers --quiet  # speeds up generation
!pip install imageio[ffmpeg] --quiet  # for video handling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 48.1 MB/s eta 0:00:00


In [2]:
# confirm GPU is working
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None - GO BACK AND ENABLE GPU!'}")

CUDA Available: True
GPU: Tesla T4


In [13]:
# Safe Authentication
from google.colab import userdata
from huggingface_hub import login
import torch
import gradio as gr
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import warnings
warnings.filterwarnings("ignore")

HF_TOKEN = userdata.get('HF_TOKEN')  # reads from Colab secrets
login(token=HF_TOKEN)
print("Authenticated with Hugging Face")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Using device: {device.upper()}")

model_id = "runwayml/stable-diffusion-v1-5"
print("Loading model...")

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=dtype,
    use_safetensors=True,
    token=HF_TOKEN
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)
if device == "cuda":
    pipe.enable_attention_slicing()

print(f"Model loaded on {device.upper()}!")

Authenticated with Hugging Face
🖥️ Using device: CUDA
⏳ Loading model...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on CUDA!


In [4]:
# Image generation function
def generate_image(
    prompt,
    negative_prompt,
    num_inference_steps,
    guidance_scale,
    image_width,
    image_height,
    seed
):
    """Generate image from text prompt using Stable Diffusion"""

    if not prompt.strip():
        return None, "Please enter a prompt!"

    # Set seed for reproducibility
    generator = None
    if seed != -1:
        generator = torch.Generator("cuda").manual_seed(int(seed))

    try:
        with torch.autocast("cuda"):
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt if negative_prompt.strip() else None,
                num_inference_steps=int(num_inference_steps),
                guidance_scale=guidance_scale,
                width=int(image_width),
                height=int(image_height),
                generator=generator
            )

        image = result.images[0]
        return image, f"Image generated successfully! Prompt: '{prompt}'"

    except Exception as e:
        return None, f"Error: {str(e)}"


# Test the function quickly
print("Generation function defined!")

Generation function defined!


In [5]:
# Video generation using animated frames from diffusion
import imageio
import numpy as np
import os

def generate_video(
    prompt,
    negative_prompt,
    num_frames,
    fps,
    guidance_scale,
    seed
):
    """Generate a short video by creating multiple frames with slight variations"""

    if not prompt.strip():
        return None, "Please enter a prompt!"

    try:
        frames = []
        num_frames = int(num_frames)

        generator = None
        if seed != -1:
            generator = torch.Generator("cuda").manual_seed(int(seed))

        # Generate frames with slight prompt variations for motion effect
        motion_prompts = [
            f"{prompt}, frame {i}, smooth motion" for i in range(num_frames)
        ]

        print(f"🎬 Generating {num_frames} frames...")

        with torch.autocast("cuda"):
            for i, frame_prompt in enumerate(motion_prompts):
                result = pipe(
                    prompt=frame_prompt,
                    negative_prompt=negative_prompt if negative_prompt.strip() else None,
                    num_inference_steps=20,   # fewer steps = faster for video
                    guidance_scale=guidance_scale,
                    width=512,
                    height=512,
                    generator=generator
                )
                frame = np.array(result.images[0])
                frames.append(frame)
                print(f"  Frame {i+1}/{num_frames} done")

        # Save as video
        video_path = "/content/generated_video.mp4"
        imageio.mimsave(video_path, frames, fps=int(fps), quality=8)

        print(f"Video saved to {video_path}")
        return video_path, f"Video generated! {num_frames} frames at {fps} FPS."

    except Exception as e:
        return None, f"Error: {str(e)}"


print("Video generation function defined!")

Video generation function defined!


In [6]:
# Build full Gradio UI
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="AI Image & Video Generator"
) as demo:

    # ── Header ──────────────────────────────────────────────
    gr.HTML("""
    <div style='text-align:center; padding: 20px;'>
        <h1 style='color:#6366f1; font-size:2.5rem;'>AI Image & Video Generator</h1>
        <p style='color:#666; font-size:1.1rem;'>
            Powered by Stable Diffusion | BTech Project | Google Colab
        </p>
    </div>
    """)

    # ── Tabs ─────────────────────────────────────────────────
    with gr.Tabs():

        # ── IMAGE GENERATION TAB ──────────────────────────────
        with gr.TabItem("Image Generation"):
            gr.Markdown("### Generate stunning images from text prompts")

            with gr.Row():
                # Left: Inputs
                with gr.Column(scale=1):
                    img_prompt = gr.Textbox(
                        label="Prompt",
                        placeholder="A futuristic city at sunset, cinematic lighting, 4K",
                        lines=3
                    )
                    img_neg_prompt = gr.Textbox(
                        label="Negative Prompt (optional)",
                        placeholder="blurry, bad quality, distorted",
                        lines=2
                    )

                    with gr.Row():
                        img_steps = gr.Slider(10, 50, value=25, step=1, label="Inference Steps")
                        img_guidance = gr.Slider(1, 20, value=7.5, step=0.5, label="Guidance Scale")

                    with gr.Row():
                        img_width = gr.Dropdown([512, 640, 768], value=512, label="Width")
                        img_height = gr.Dropdown([512, 640, 768], value=512, label="Height")

                    img_seed = gr.Number(value=-1, label="Seed (-1 = random)")
                    img_btn = gr.Button("Generate Image", variant="primary", size="lg")

                # Right: Output
                with gr.Column(scale=1):
                    img_output = gr.Image(label="Generated Image", type="pil")
                    img_status = gr.Textbox(label="Status", interactive=False)

            # Examples
            gr.Examples(
                examples=[
                    ["A majestic lion in a golden savanna, photorealistic, 8K", "blurry, cartoon", 25, 7.5, 512, 512, 42],
                    ["Cyberpunk Tokyo street at night, neon lights, rain", "ugly, distorted", 30, 8.0, 512, 512, -1],
                    ["A beautiful mountain landscape with snow, sunset, oil painting", "", 25, 7.5, 512, 512, 123],
                    ["Astronaut floating in space, Earth in background, realistic", "cartoon, blurry", 25, 7.5, 512, 512, -1],
                ],
                inputs=[img_prompt, img_neg_prompt, img_steps, img_guidance, img_width, img_height, img_seed],
                outputs=[img_output, img_status],
                fn=generate_image,
                cache_examples=False,
                label="Try These Examples"
            )

            img_btn.click(
                fn=generate_image,
                inputs=[img_prompt, img_neg_prompt, img_steps, img_guidance, img_width, img_height, img_seed],
                outputs=[img_output, img_status]
            )

        # ── VIDEO GENERATION TAB ──────────────────────────────
        with gr.TabItem("Video Generation"):
            gr.Markdown("### Generate short AI videos from text prompts")
            gr.Markdown("> Video generation creates multiple frames — each frame takes ~10-15 seconds on T4 GPU. Keep frames low (3-5) for demo.")

            with gr.Row():
                with gr.Column(scale=1):
                    vid_prompt = gr.Textbox(
                        label="Prompt",
                        placeholder="A phoenix rising from flames, dramatic, cinematic",
                        lines=3
                    )
                    vid_neg_prompt = gr.Textbox(
                        label="Negative Prompt (optional)",
                        placeholder="blurry, static, bad quality",
                        lines=2
                    )

                    with gr.Row():
                        vid_frames = gr.Slider(2, 10, value=4, step=1, label="Number of Frames")
                        vid_fps = gr.Slider(2, 12, value=4, step=1, label="FPS")

                    vid_guidance = gr.Slider(1, 20, value=7.5, step=0.5, label="Guidance Scale")
                    vid_seed = gr.Number(value=42, label="Seed (-1 = random)")
                    vid_btn = gr.Button("🎬 Generate Video", variant="primary", size="lg")

                with gr.Column(scale=1):
                    vid_output = gr.Video(label="Generated Video")
                    vid_status = gr.Textbox(label="Status", interactive=False)

            gr.Examples(
                examples=[
                    ["A dragon flying over mountains, fantasy, epic", "blurry", 4, 4, 7.5, 42],
                    ["Ocean waves crashing on rocks, golden hour", "", 4, 4, 7.5, -1],
                ],
                inputs=[vid_prompt, vid_neg_prompt, vid_frames, vid_fps, vid_guidance, vid_seed],
                outputs=[vid_output, vid_status],
                fn=generate_video,
                cache_examples=False,
                label="Try These Examples"
            )

            vid_btn.click(
                fn=generate_video,
                inputs=[vid_prompt, vid_neg_prompt, vid_frames, vid_fps, vid_guidance, vid_seed],
                outputs=[vid_output, vid_status]
            )

        # ── ABOUT TAB ─────────────────────────────────────────
        with gr.TabItem("About"):
            gr.Markdown("""
            ## About This Project

            ### BTech 3rd Year AI/ML Project

            **Project:** AI-Based Image & Video Generation using Diffusion Models

            **Tech Stack:**
            - **Hugging Face Diffusers** — Stable Diffusion pipeline
            - **Gradio** — Interactive web UI
            - **PyTorch** — Deep learning framework
            - **Google Colab** — Free GPU (NVIDIA T4)

            **How It Works:**
            - **Image Generation** uses Stable Diffusion 2.1 to generate images from text prompts via the denoising diffusion process
            - **Video Generation** creates sequential frames using the same model and stitches them into an MP4 using imageio

            **Key Concepts:**
            - Text-to-Image using Latent Diffusion Models
            - CLIP text encoder for prompt understanding
            - DPM-Solver++ scheduler for fast inference
            - Classifier-Free Guidance (CFG scale)

            **Model Used:** `stabilityai/stable-diffusion-2-1-base`

            ---
            *Built with using open-source AI tools*
            """)

# ── Launch ────────────────────────────────────────────────────
print("Launching Gradio app...")
demo.launch(
    share=True,          # Creates a public shareable link
    debug=False,
    show_error=True
)

Launching Gradio app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bc608cfea4074157c1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
# Quick test
test_image, status = generate_image(
    prompt="A beautiful sunset over the ocean, photorealistic",
    negative_prompt="blurry, bad quality",
    num_inference_steps=20,
    guidance_scale=7.5,
    image_width=512,
    image_height=512,
    seed=42
)

if test_image:
    print(status)
    test_image.show()
else:
    print(status)

  0%|          | 0/20 [00:00<?, ?it/s]

✅ Image generated successfully! Prompt: 'A beautiful sunset over the ocean, photorealistic'


In [11]:
# Mount Google Drive to save outputs
from google.colab import drive
drive.mount('/content/drive')

import shutil
from datetime import datetime

def save_to_drive(image, filename_prefix="generated"):
    if image is not None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = f"/content/drive/MyDrive/AI_Generated/{filename_prefix}_{timestamp}.png"
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        image.save(save_path)
        print(f"Saved to Google Drive: {save_path}")
    else:
        print("No image to save")

Mounted at /content/drive
